### K-Nearest Neighbours

In [11]:
from scipy.sparse import load_npz
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# 1) Cargamos las matrices de usuarios-items, tanto de train como de test: 
path = '../data/processed/'

train_set = load_npz(path+'train_set.npz')
test_set = load_npz(path+'test_set.npz')

In [19]:
# 2) Definimos una función para calcular similitud entre usuarios. 
# Aplicaremos la similitud coseno por nuestro conocimiento sobre él al haberla usado en PLN
# y debido a sus buenos resultados: 

def cosine_similarity_user_all(user_idx, matrix):
    """
    Calcula similitud coseno entre un usuario y todos los demás.
    """
    # Extraemos vector del usuario: 
    user_vec = matrix[user_idx]
    
    # Calculamos norma L2 del usuario: 
    user_norm = np.sqrt(np.array(user_vec.power(2).sum()))
    if user_norm == 0:
        user_norm = 1.0
    
    # Calculamos normas L2 de todos los usuarios: 
    all_norms = np.sqrt(np.array(matrix.power(2).sum(axis=1)).flatten())
    all_norms[all_norms == 0] = 1.0
    
    # Aplicamos el producto escalar de un usuario con todos los demás de forma vectorizada: 
    dot_products = (matrix @ user_vec.T).toarray().flatten()
    
    # Dividimos por normas: 
    similarities = dot_products / (user_norm * all_norms)
    
    return similarities

In [20]:
# 3) Definimos una función para obtener los k vecinos más similares con reespecto a un usuario: 

def get_k_neighbours(user_idx, matrix, k=10):
    """
    Encuentra los k vecinos más similares a un usuario.
    """
    # Calculamos similitudes coseno usando la función creada previamente: 
    similarities = cosine_similarity_user_all(user_idx, matrix)
    
    # Ignoramos similitud consigo mismo: 
    similarities[user_idx] = -1.0
    
    # Encontramos los k índices con mayor similitud usando argpartition, 
    # que permite encontrar estos k índices sin ordenar todo: 
    top_k_indices = np.argpartition(similarities, -k)[-k:]
    top_k_similarities = similarities[top_k_indices]
    
    # Ordenamos descendentemente solo los k elementos: 
    sort_order = np.argsort(top_k_similarities)[::-1]
    
    return top_k_indices[sort_order], top_k_similarities[sort_order]


# Prueba de funcionamiento:
vecinos_ids, similitudes = get_k_neighbours(user_idx=150, matrix=train_set, k=10)
vecinos_ids, similitudes

(array([ 36749, 526568, 750420, 669865,  18962, 417239, 645945, 182589,
        598055, 704936]),
 array([0.76471911, 0.76471911, 0.68213195, 0.66560255, 0.66226618,
        0.6439113 , 0.63523581, 0.6104203 , 0.59948587, 0.54073807]))